<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session04-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 4 — In-class discussion problem (3 of 3)

Work this out **by hand in your group first** — then run the code cells to check your answers before presenting.

## Guide trees: when do they matter, and are they phylogenies?

**Part 1.** Two progressive alignments of the same sequences are built from
two different guide trees, so they merge the sequences in a different
order. When would you expect the final alignments to differ substantially,
and when would the order barely matter? Think of two cases:

- **A:** a few near-identical sequences (all > 95% identical);
- **B:** a few divergent protein domains (all around 40% different from each other).

**Part 2.** A guide tree is usually built quickly, by **UPGMA**: repeatedly
merge the two closest groups, averaging their distances. That implicitly
assumes all lineages evolve at the same rate (a *molecular clock*). Below
are four sequences whose true history is ((A, B), (C, D)), but **B has
evolved much faster** than the others. Before running anything: which
pair will UPGMA merge first? Will its tree show A and B as relatives?

**Part 3.** Why should a guide tree not automatically be read as a
phylogeny?

In [1]:
import numpy as np
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# Part 1: toy distance matrices (fraction of differing residues)
D_similar = np.array([[0.000, 0.020, 0.021, 0.019],
                      [0.020, 0.000, 0.022, 0.018],
                      [0.021, 0.022, 0.000, 0.020],
                      [0.019, 0.018, 0.020, 0.000]])
D_divergent = np.array([[0.00, 0.40, 0.45, 0.42],
                        [0.40, 0.00, 0.38, 0.44],
                        [0.45, 0.38, 0.00, 0.41],
                        [0.42, 0.44, 0.41, 0.00]])
for name, D in [("A, near-identical", D_similar), ("B, divergent", D_divergent)]:
    Z = linkage(squareform(D, checks=False), method="average")    # UPGMA
    off = D[np.triu_indices(4, 1)]
    print(f"{name:18s} pairwise distances {off.min():.3f}-{off.max():.3f}; first merge: sequences "
          f"{int(Z[0, 0])} and {int(Z[0, 1])} at distance {Z[0, 2]:.3f}")

A, near-identical  pairwise distances 0.018-0.022; first merge: sequences 1 and 3 at distance 0.018
B, divergent       pairwise distances 0.380-0.450; first merge: sequences 1 and 2 at distance 0.380


In [2]:
# Part 2: true tree ((A:0.1, B:0.6):0.1, (C:0.1, D:0.1)); distances are the path lengths along it
from Bio import Phylo
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor
names = ["A", "B", "C", "D"]
dm = DistanceMatrix(names, [[0], [0.7, 0], [0.3, 0.8, 0], [0.3, 0.8, 0.2, 0]])
print("distances: A-B 0.7, A-C 0.3, A-D 0.3, B-C 0.8, B-D 0.8, C-D 0.2\n")
for method in ["upgma", "nj"]:
    tree = getattr(DistanceTreeConstructor(), method)(dm)
    for c in tree.get_nonterminals():
        c.name = None
    print(method.upper(), "tree:")
    Phylo.draw_ascii(tree)

distances: A-B 0.7, A-C 0.3, A-D 0.3, B-C 0.8, B-D 0.8, C-D 0.2

UPGMA tree:
  ___________________________________________________________________________ B
 |
_|                                                       ____________________ D
 |                                             _________|
 |____________________________________________|         |____________________ C
                                              |
                                              |______________________________ A

NJ tree:
            __________ A
  _________|
 |         |_________________________________________________________________ B
_|
 |_________ C
 |
 |_________ D



**Discussion points.**

*Part 1.* In case A every pair is almost identical, so whatever order the
guide tree picks, each merge aligns sequences that differ in only a few
positions, and the result barely changes. In case B every pair is
substantially different, an early merge can lock a doubtful gap placement
in place, and progressive alignment never revisits it: the order can
change the result. Iterative refinement and consistency-based methods
exist to reduce exactly this dependence.

*Part 2.* UPGMA merges the closest pair first, C and D (0.2), then adds A
(0.3 away), and B comes last: its tree says A is closer to C and D than to
B, which is wrong. B is far from everything only because it evolved fast.
Neighbour joining corrects for unequal rates and recovers A and B as
relatives (look at the long branch it gives B).

*Part 3.* A guide tree is an ordering device: it only has to put similar
sequences together early enough for a good alignment. A phylogeny is a
hypothesis about descent, and needs methods (and checks, such as
bootstrap support) built for that purpose. One group presents.